# Offline Kaggle GPU pilot (Qwen2.5-0.5B + LoRA)
This experimental notebook fine-tunes an attached **local** base model as a three-class preference predictor, evaluates a held-out split and writes a submission. The model and Kaggle data must be attached as **Inputs**. This notebook has **not yet been tested on Kaggle GPU**.

1. Accept [competition rules](https://www.kaggle.com/competitions/llm-classification-finetuning). Attach the competition dataset as Input.
2. Attach a legitimately accessible, publicly usable *complete* Qwen2.5-0.5B-Instruct base-model input (model weights, config and tokenizer). Set the exact path in the last cell.
3. Turn GPU on; **turn Internet off** for code submission. Configure `transformers`, `peft`, `accelerate`, `sentencepiece` via the Kaggle Dependency Manager / pre-attached wheels if they are absent.
4. Pilot uses 4,000 training examples plus A/B reversal; *not a full training claim*. It writes `/kaggle/working/submission.csv` and saves only your derived adapter/metrics under `/kaggle/working/gpu_pilot/`.
5. Check that the final committed version completes within Kaggle's nine-hour GPU limit on the **hidden ~25K-row test**; the tiny preview test is not representative. Avoid publishing the competition's raw data.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Switch Kaggle Notebook accelerator to GPU.')
import transformers, peft, accelerate
print('transformers', transformers.__version__, 'peft', peft.__version__)


In [ ]:
# Generated by python scripts/sync_gpu_notebook.py; do not edit this cell by hand.
from pathlib import Path
import sys
source_dir = Path('/kaggle/working/src')
source_dir.mkdir(parents=True, exist_ok=True)
(source_dir / '__init__.py').write_text('', encoding='utf-8')
(source_dir / 'baseline.py').write_text("\"\"\"Leakage-controlled, swap-augmented TF-IDF baseline for Kaggle LLM preference prediction.\n\nThis is a classical ML baseline, not an LLM fine-tuning run.\n\"\"\"\nimport argparse\nimport ast\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack, vstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nTARGETS = [\"winner_model_a\", \"winner_model_b\", \"winner_tie\"]\nTEXT_COLUMNS = [\"prompt\", \"response_a\", \"response_b\"]\n\n\ndef flatten_messages(value, max_chars=2400):\n    \"\"\"Normalize Kaggle's serialized lists of turns; cap length for a CPU starter.\"\"\"\n    if value is None or (isinstance(value, float) and np.isnan(value)):\n        return \"\"\n    if isinstance(value, str):\n        text = value.strip()\n        if text.startswith(\"[\"):\n            try:\n                value = json.loads(text)\n            except (ValueError, TypeError):\n                try:\n                    value = ast.literal_eval(text)\n                except (ValueError, SyntaxError):\n                    value = text\n        else:\n            value = text\n    if isinstance(value, (list, tuple)):\n        text = \" \".join(\"\" if item is None else str(item) for item in value)\n    else:\n        text = str(value)\n    return text[:max_chars]\n\n\ndef normalized_frame(df):\n    missing = [c for c in TEXT_COLUMNS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing text columns: {missing}\")\n    return pd.DataFrame(\n        {col: [flatten_messages(v) for v in df[col]] for col in TEXT_COLUMNS},\n        index=df.index,\n    )\n\n\ndef get_labels(df):\n    missing = [c for c in TARGETS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing label columns: {missing}\")\n    y = df[TARGETS].to_numpy(dtype=int)\n    if not np.all(y.sum(axis=1) == 1) or not np.all((y == 0) | (y == 1)):\n        raise ValueError(\"Expected exactly one binary winner label per training row\")\n    return y.argmax(axis=1)\n\n\ndef flip_pairs(df):\n    flipped = df.copy()\n    flipped[\"response_a\"], flipped[\"response_b\"] = (\n        df[\"response_b\"].copy(), df[\"response_a\"].copy()\n    )\n    return flipped\n\n\ndef make_vectorizer(df):\n    # Only fit on training-partition texts; do not fit on held-out validation/test.\n    min_df = 2 if len(df) >= 30 else 1\n    vectorizer = TfidfVectorizer(\n        ngram_range=(1, 2), max_features=35000, min_df=min_df,\n        strip_accents=\"unicode\", sublinear_tf=True, dtype=np.float32,\n    )\n    vectorizer.fit(\n        df[\"prompt\"].tolist() + df[\"response_a\"].tolist() +\n        df[\"response_b\"].tolist()\n    )\n    return vectorizer\n\n\ndef pair_features(df, vectorizer):\n    q = vectorizer.transform(df[\"prompt\"])\n    a = vectorizer.transform(df[\"response_a\"])\n    b = vectorizer.transform(df[\"response_b\"])\n    len_a = df[\"response_a\"].str.len().to_numpy(dtype=np.float32)\n    len_b = df[\"response_b\"].str.len().to_numpy(dtype=np.float32)\n    len_q = df[\"prompt\"].str.len().to_numpy(dtype=np.float32)\n    numeric = np.column_stack([\n        np.log1p(len_a) - np.log1p(len_b),\n        (np.log1p(len_a) + np.log1p(len_b)) / 2,\n        np.log1p(len_q),\n    ]) / 10.0\n    return hstack([q, a - b, (a + b) * 0.5, csr_matrix(numeric)],\n                  format=\"csr\", dtype=np.float32)\n\n\ndef fit_baseline(df, y):\n    vectorizer = make_vectorizer(df)\n    x_original = pair_features(df, vectorizer)\n    x_flipped = pair_features(flip_pairs(df), vectorizer)\n    swapped_labels = np.where(y == 0, 1, np.where(y == 1, 0, 2))\n    model = LogisticRegression(C=2.0, max_iter=300, random_state=42)\n    model.fit(vstack([x_original, x_flipped], format=\"csr\"),\n              np.concatenate([y, swapped_labels]))\n    return {\"vectorizer\": vectorizer, \"model\": model, \"targets\": TARGETS}\n\n\ndef predict_prob(bundle, df):\n    features = pair_features(df, bundle[\"vectorizer\"])\n    raw = bundle[\"model\"].predict_proba(features)\n    out = np.zeros((len(df), len(TARGETS)), dtype=np.float64)\n    for col_idx, class_idx in enumerate(bundle[\"model\"].classes_):\n        out[:, int(class_idx)] = raw[:, col_idx]\n    return out / out.sum(axis=1, keepdims=True)\n\n\ndef train(train_csv, out_dir, validation_fraction=0.15):\n    raw = pd.read_csv(train_csv)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if len(np.unique(y)) != 3 or np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class must have at least two examples for validation\")\n    x_tr, x_val, y_tr, y_val = train_test_split(\n        df, y, test_size=validation_fraction, random_state=42, stratify=y\n    )\n    validation_bundle = fit_baseline(x_tr, y_tr)\n    val_probs = predict_prob(validation_bundle, x_val)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_probs, labels=[0, 1, 2])),\n        \"train_rows\": int(len(x_tr)),\n        \"validation_rows\": int(len(x_val)),\n        \"full_rows\": int(len(df)),\n        \"seed\": 42,\n        \"note\": \"Random stratified split; not a competition leaderboard result.\",\n    }\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \"validation_metrics.json\").write_text(\n        json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    # Refit using all labeled data only after holding out validation above.\n    joblib.dump(fit_baseline(df, y), output / \"baseline.joblib\")\n    return metrics\n\n\ndef predict(test_csv, model_path, out_csv):\n    test = pd.read_csv(test_csv)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Test CSV must contain id\")\n    bundle = joblib.load(model_path)  # Load only artifacts you created/trust.\n    probabilities = predict_prob(bundle, normalized_frame(test))\n    result = pd.DataFrame(probabilities, columns=TARGETS)\n    result.insert(0, \"id\", test[\"id\"])\n    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(out_csv, index=False)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\"command\", required=True)\n    fit = sub.add_parser(\"train\")\n    fit.add_argument(\"--train\", default=\"data/train.csv\")\n    fit.add_argument(\"--out\", default=\"artifacts\")\n    infer = sub.add_parser(\"predict\")\n    infer.add_argument(\"--test\", default=\"data/test.csv\")\n    infer.add_argument(\"--model\", default=\"artifacts/baseline.joblib\")\n    infer.add_argument(\"--out\", default=\"submission.csv\")\n    args = parser.parse_args()\n    if args.command == \"train\":\n        print(json.dumps(train(args.train, args.out), indent=2))\n    else:\n        print(f\"Wrote {len(predict(args.test, args.model, args.out))} rows: {args.out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
(source_dir / 'finetune_lora.py').write_text("\"\"\"Optional GPU pilot: fine-tune an offline Qwen2.5-0.5B sequence classifier with LoRA.\n\nOnly run after attaching legitimately accessible model weights and official Kaggle\ncompetition data. This is a pilot; no actual GPU experiment is claimed here.\n\nWith offline Kaggle submissions, attach the *complete* base model directory\n(config, tokenizer and weights) as an Input; never fetch from Hugging Face at runtime.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, flip_pairs, get_labels, normalized_frame\n\n\ndef render_pair(row):\n    \"\"\"Fixed prompt template and truncation; do not include train-only model names.\"\"\"\n    question = flatten_messages(row[\"prompt\"], max_chars=1200)\n    a = flatten_messages(row[\"response_a\"], max_chars=2400)\n    b = flatten_messages(row[\"response_b\"], max_chars=2400)\n    return (\n        \"A human gave two chatbots the same user request.\\n\"\n        f\"User request: {question}\\n\"\n        f\"Response A: {a}\\n\"\n        f\"Response B: {b}\\n\"\n        \"Predict whether the human prefers response A, response B, or a tie.\"\n    )\n\n\ndef swap_labels(labels):\n    labels = np.asarray(labels, dtype=np.int64)\n    if not np.isin(labels, [0, 1, 2]).all():\n        raise ValueError(\"Expected A=0, B=1, tie=2\")\n    return np.where(labels == 0, 1, np.where(labels == 1, 0, 2))\n\n\ndef train_and_predict(args):\n    # Lazy imports ensure that unit tests do not need GPU-only dependencies.\n    import torch\n    from peft import LoraConfig, TaskType, get_peft_model\n    from torch.utils.data import Dataset\n    from transformers import (\n        AutoModelForSequenceClassification, AutoTokenizer,\n        DataCollatorWithPadding, Trainer, TrainingArguments,\n    )\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"This LoRA pilot requires a CUDA GPU; use the CPU baseline otherwise.\")\n    model_dir = Path(args.base_model).expanduser()\n    if not (model_dir / \"config.json\").exists():\n        raise FileNotFoundError(\n            \"Supply the complete offline base model directory via --base-model; \"\n            f\"no config.json found in {model_dir}\"\n        )\n    if args.pilot_rows != 0 and args.pilot_rows < 30:\n        raise ValueError(\"pilot_rows must be 0 (all rows) or at least 30\")\n\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    raw = pd.read_csv(args.train)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class needs at least two rows\")\n    x_train, x_val, y_train, y_val = train_test_split(\n        df, y, test_size=0.15, stratify=y, random_state=args.seed\n    )\n    # Cap *training only*. Keep validation untouched for honest pilot comparison.\n    if args.pilot_rows and len(x_train) > args.pilot_rows:\n        x_train, _, y_train, _ = train_test_split(\n            x_train, y_train, train_size=args.pilot_rows,\n            stratify=y_train, random_state=args.seed\n        )\n    if args.swap_train:\n        x_original, y_original = x_train.copy(), y_train.copy()\n        x_train = pd.concat(\n            [x_original, flip_pairs(x_original)], ignore_index=True\n        )\n        y_train = np.concatenate([y_original, swap_labels(y_original)])\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        model_dir, local_files_only=True, trust_remote_code=False\n    )\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer has neither pad nor EOS token\")\n        tokenizer.pad_token = tokenizer.eos_token\n\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_dir, num_labels=3, torch_dtype=dtype,\n        local_files_only=True, trust_remote_code=False,\n    )\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    model = get_peft_model(\n        model,\n        LoraConfig(\n            task_type=TaskType.SEQ_CLS,\n            r=8, lora_alpha=16, lora_dropout=0.05,\n            target_modules=[\"q_proj\", \"v_proj\"],\n            modules_to_save=[\"score\"],\n        ),\n    )\n\n    class PairDataset(Dataset):\n        def __init__(self, frame, labels=None):\n            self.texts = [render_pair(row) for row in frame.to_dict(\"records\")]\n            self.labels = labels\n\n        def __len__(self):\n            return len(self.texts)\n\n        def __getitem__(self, i):\n            encoded = tokenizer(\n                self.texts[i], truncation=True, max_length=args.max_length\n            )\n            if self.labels is not None:\n                encoded[\"labels\"] = int(self.labels[i])\n            return encoded\n\n    output = Path(args.output)\n    output.mkdir(parents=True, exist_ok=True)\n    config = TrainingArguments(\n        output_dir=str(output / \"trainer\"),\n        num_train_epochs=args.epochs,\n        per_device_train_batch_size=args.batch_size,\n        per_device_eval_batch_size=args.eval_batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.learning_rate,\n        weight_decay=0.01,\n        lr_scheduler_type=\"cosine\",\n        warmup_ratio=0.05,\n        fp16=dtype == torch.float16,\n        bf16=dtype == torch.bfloat16,\n        eval_strategy=\"no\",\n        save_strategy=\"no\",\n        logging_strategy=\"steps\",\n        logging_steps=25,\n        report_to=\"none\",\n        remove_unused_columns=False,\n        dataloader_num_workers=0,\n        seed=args.seed,\n    )\n    trainer = Trainer(\n        model=model,\n        args=config,\n        train_dataset=PairDataset(x_train, y_train),\n        data_collator=DataCollatorWithPadding(\n            tokenizer=tokenizer, pad_to_multiple_of=8\n        ),\n        processing_class=tokenizer,\n    )\n    trainer.train()\n    val_logits = trainer.predict(PairDataset(x_val)).predictions\n    if isinstance(val_logits, tuple):\n        val_logits = val_logits[0]\n    val_prob = softmax(np.asarray(val_logits, dtype=np.float64), axis=-1)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_prob, labels=[0, 1, 2])),\n        \"validation_rows\": len(x_val),\n        \"train_rows_after_augmentation\": len(x_train),\n        \"pilot_rows\": args.pilot_rows,\n        \"seed\": args.seed,\n        \"max_length_tokens\": args.max_length,\n        \"base_model_dir\": model_dir.name,\n        \"training_type\": \"Qwen2.5-0.5B sequence classification head + LoRA\",\n        \"caution\": \"Preliminary pilot; independent replication and full-data run pending.\",\n    }\n    # Probe original/swap consistency on a bounded held-out subset.\n    subset = x_val.iloc[: min(128, len(x_val))]\n    original_logits = trainer.predict(PairDataset(subset)).predictions\n    swapped_logits = trainer.predict(PairDataset(flip_pairs(subset))).predictions\n    if isinstance(original_logits, tuple):\n        original_logits = original_logits[0]\n    if isinstance(swapped_logits, tuple):\n        swapped_logits = swapped_logits[0]\n    original_prob = softmax(np.asarray(original_logits, dtype=np.float64), axis=-1)\n    swapped_prob = softmax(np.asarray(swapped_logits, dtype=np.float64), axis=-1)[:, [1, 0, 2]]\n    metrics[\"swap_probe_mean_abs_difference\"] = float(\n        np.abs(original_prob - swapped_prob).mean()\n    )\n    # Preserve successful validation metrics even if adapter saving or optional\n    # 25K-row Kaggle test inference fails. The JSON is aggregate-only.\n    metrics[\"pipeline_status\"] = \"validation_completed\"\n    try:\n        adapter_dir = output / \"adapter\"\n        trainer.model.save_pretrained(adapter_dir)\n        tokenizer.save_pretrained(adapter_dir)\n        metrics[\"pipeline_status\"] = \"adapter_saved\"\n\n        if args.test:\n            test = pd.read_csv(args.test)\n            if \"id\" not in test.columns:\n                raise ValueError(\"Test data require id column\")\n            test_frame = normalized_frame(test)\n            test_logits = trainer.predict(PairDataset(test_frame)).predictions\n            if isinstance(test_logits, tuple):\n                test_logits = test_logits[0]\n            probs = softmax(np.asarray(test_logits, dtype=np.float64), axis=-1)\n            submission = pd.DataFrame(probs, columns=TARGETS)\n            submission.insert(0, \"id\", test[\"id\"])\n            submission_path = Path(args.submission)\n            submission_path.parent.mkdir(parents=True, exist_ok=True)\n            submission.to_csv(submission_path, index=False)\n            metrics[\"submission_rows\"] = int(len(submission))\n            metrics[\"pipeline_status\"] = \"submission_completed\"\n            print(f\"Submission written to {submission_path}\")\n    except Exception as exc:\n        metrics[\"pipeline_status\"] = \"downstream_failed\"\n        metrics[\"downstream_error_type\"] = type(exc).__name__\n        raise\n    finally:\n        (output / \"gpu_pilot_metrics.json\").write_text(\n            json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n        )\n    print(json.dumps(metrics, indent=2))\n    return metrics\n\n\ndef parse_args():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--train\", default=\"data/train.csv\")\n    p.add_argument(\"--test\", default=None)\n    p.add_argument(\"--base-model\", required=True,\n                   help=\"Complete *local* Qwen2.5-0.5B-Instruct weights/tokenizer folder\")\n    p.add_argument(\"--output\", default=\"artifacts/gpu_pilot\")\n    p.add_argument(\"--submission\", default=\"submission.csv\")\n    p.add_argument(\"--pilot-rows\", type=int, default=4000,\n                   help=\"Training cap excluding held-out validation; 0 uses all train rows\")\n    p.add_argument(\"--max-length\", type=int, default=384)\n    p.add_argument(\"--epochs\", type=float, default=1.0)\n    p.add_argument(\"--batch-size\", type=int, default=2)\n    p.add_argument(\"--eval-batch-size\", type=int, default=4)\n    p.add_argument(\"--grad-accum\", type=int, default=8)\n    p.add_argument(\"--learning-rate\", type=float, default=2e-4)\n    p.add_argument(\"--seed\", type=int, default=42)\n    p.add_argument(\"--no-swap-train\", action=\"store_false\", dest=\"swap_train\")\n    p.set_defaults(swap_train=True)\n    return p.parse_args()\n\n\nif __name__ == \"__main__\":\n    train_and_predict(parse_args())\n", encoding='utf-8')
sys.path.insert(0, '/kaggle/working')
from src.finetune_lora import train_and_predict


In [ ]:
from argparse import Namespace
from pathlib import Path
INPUT = Path('/kaggle/input/llm-classification-finetuning')
# CHANGE this to your attached model directory that contains config.json
BASE = Path('/kaggle/input/YOUR_OFFLINE_QWEN_MODEL')
assert (INPUT / 'train.csv').exists(), 'Attach the official competition dataset as Input'
assert (BASE / 'config.json').exists(), 'Set BASE to the attached model folder'
args = Namespace(
    train=str(INPUT / 'train.csv'), test=str(INPUT / 'test.csv'),
    base_model=str(BASE), output='/kaggle/working/gpu_pilot',
    submission='/kaggle/working/submission.csv',
    pilot_rows=4000, max_length=384, epochs=1.0, batch_size=2,
    eval_batch_size=4, grad_accum=8, learning_rate=2e-4,
    seed=42, swap_train=True,
)
result = train_and_predict(args)
print('Pilot validation log loss:', result['validation_log_loss'])
print('Submission:', args.submission)
